In [1]:
import os
from math import nan
import pandas as pd
pd.options.plotting.backend='plotly'

# Grid Search Results

```
Model                   SkillDaily  SkillsPos
u24-96_d0.1_n48_f4      0.435722    0.611111
u48-24_d0.1_n24_f4      0.415693    0.594444
u128-96_d0.1_n24_f4     0.413271    0.605556
```

In [2]:
#results_dir = r'results/redcliff_healthcenter/gridsearch v1.7 250324'
results_dir = r'results/jpl_ev/hpsearch_v1-8_250326/'

results = pd.DataFrame({'Model':[],'Skill':[],'Daily Skills Pos':[]})

for model in [x for x in os.listdir(results_dir) if x[0] == 'u']:
    model_dir = results_dir + r'/' + model
    
    if 'all_forecasts.csv' in os.listdir(model_dir):

        df = pd.read_csv(model_dir + r'/all_forecasts.csv',index_col=0)

        df['ErrorPred'] = (df.Pred - df.Load)/df.Load.max()
        df['ErrorPers'] = (df.Persist - df.Load)/df.Load.max()

        mae_pers_all = df[df.ErrorPers.abs()>0.0].ErrorPred.abs().mean()
        mae_pred_all = df[df.ErrorPers.abs()>0.0].ErrorPers.abs().mean()
        
        grouped = df.groupby('timestamp_update')
        skills = []
        for name, group in grouped:
            mae_pers = group['ErrorPers'].abs().mean()
            mae_pred = group['ErrorPred'].abs().mean()
            if mae_pers > 0:
                skill = 1 - mae_pred / mae_pers
            else:
                skill = nan
            skills.append(skill)
        daily_skills_pos = len([x for x in skills if x > 0])/len(skills)
        
    else:
        mae_pers = nan
        mae_pred = nan
        daily_skills_pos = nan
    
    results.loc[len(results)] = {'Model':model,'Skill':1 - mae_pred_all / mae_pers_all,'Daily Skills Pos':daily_skills_pos}
        
results.sort_values('Skill',ascending=False)

,Model,Skill,Daily Skills Pos
336,u48-48_d0_n384_f3,0.715081,0.000000
338,u48-48_d0_n480_f4,0.715081,NaN
337,u48-48_d0_n480_f3,0.715081,NaN
438,u96-48_d0_n192_f1,0.706265,0.000000
420,u96-24_d0.1_n192_f4,0.705125,0.000000
...,...,...,...
122,u128-96_d0_n96_f1,-0.538680,0.641667
339,u48-48_d0_n96_f2,-0.541994,0.708333
213,u256-256_d0_n480_f2,-0.559221,NaN
212,u256-256_d0.1_n96_f4,-0.559221,0.661111
